# 00 · Formas y normalización con NumPy — inicio para completar

Normalizar por columna, preservar formas y resolver una columna constante sin dividir por cero.

**Ideas que aparecen:** funciones, listas, media y desviación (cápsulas 1 y 4). Puedes consultar las cápsulas
opcionales de `ruta/Puentes de entrada.md` cuando alguna te haga falta.

Datos **sintéticos educativos**, creados en este repositorio y dedicados a
CC0-1.0. No representan personas ni un rendimiento oficial IOAI.
Todo el ejercicio usa CPU y archivos locales; no requiere cuentas ni red.

El inicio ya corre. Las celdas de experimentación son lugares para cambiar una idea y observar qué ocurre; la solución está en otro archivo si quieres contrastarla.
Puedes recorrerlo en una o varias sesiones. No hay límite de juez ni
obligación de completar todos los experimentos para abrir el siguiente cuaderno.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (10, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})

DATOS = Path("datos")
assert DATOS.is_dir(), "Abre el notebook desde su carpeta: deben existir datos/ e inicio.ipynb."
SEMILLA = 17


## Antes de escribir
Una fila es un ejemplo y una columna una variable. Para `(n, d)`,
`mean(axis=0)` tiene forma `(d,)`. Estandarizar aquí significa restar
la media de **cada columna** y dividir por su desviación poblacional
(`ddof=0`). Una columna constante se transforma en ceros: su media
será cero y su desviación también, no uno.

Este laboratorio es una operación sobre una matriz. Cuando haya
entrenamiento y validación, aprende esas estadísticas solo en
entrenamiento y reutilízalas (laboratorio 04).


In [ ]:
from time import perf_counter
X = np.loadtxt(DATOS / "matriz.csv", delimiter=",", skiprows=1)
assert X.shape == (12, 3)
print("Forma:", X.shape, "media por columna:", X.mean(axis=0))

def normalizar_lento(X):
    X = np.asarray(X, dtype=float)
    salida = np.empty_like(X)
    for j in range(X.shape[1]):
        media = sum(float(v) for v in X[:, j]) / len(X)
        desv = (sum((float(v) - media) ** 2 for v in X[:, j]) / len(X)) ** 0.5
        for i in range(len(X)):
            salida[i, j] = (X[i, j] - media) / desv if desv > 0 else 0.0
    return salida


## Zona de experimentación
Sustituye el cuerpo de `normalizar` por operaciones sobre arreglos,
sin ciclos. Antes de ejecutar, anota las formas de media, desviación
y salida. El inicio delega en el baseline correcto y permite correr
el notebook entero; todavía no has hecho la vectorización.


In [ ]:
def normalizar(X):
    # TRABAJO: reemplaza esta delegación por operaciones de NumPy.
    return normalizar_lento(X)


## Comparación en las mismas condiciones
Compara ambas funciones sobre exactamente la misma matriz. La solución de referencia conserva forma, produce valores finitos y coincide hasta error ≤ 1e-10; la velocidad es una observación del equipo, no una nota ni una garantía de 20×.


In [ ]:
esperado = normalizar_lento(X)
obtenido = normalizar(X)
error = float(np.max(np.abs(esperado - obtenido)))
assert obtenido.shape == X.shape and np.isfinite(obtenido).all()
# Matriz pequeña suficiente para medir sin convertirlo en una espera larga.
grande = np.tile(X, (300, 1))
tiempos = {}
for nombre, funcion in [("baseline", normalizar_lento), ("propuesta", normalizar)]:
    funcion(grande)  # calentamiento
    mediciones = []
    for _ in range(3):
        inicio_reloj = perf_counter()
        funcion(grande)
        mediciones.append(perf_counter() - inicio_reloj)
    tiempos[nombre] = float(np.median(mediciones))
resultado = {"baseline": 0.0, "validacion": error, "tiempos_segundos": tiempos}
print("Error máximo:", error, "tiempos:", tiempos)


<details><summary>Idea · eje</summary>Al reducir el eje 0 desaparecen las filas; queda una estadística por columna.</details>
<details><summary>Idea · constantes</summary>Usa un divisor de 1 donde la desviación sea 0. El numerador ya es 0.</details>
<details><summary>Idea · copia</summary>Devuelve un arreglo nuevo. Comprueba que la entrada no cambió.</details>


## Transferencia: decide antes de mirar el resultado
La segunda matriz tiene dos columnas y una es constante. Predice su forma y desviaciones antes de correr. Prueba además una sola fila. Explica por qué pedir desviación uno para la constante sería imposible.


In [ ]:
nuevo = np.loadtxt(DATOS / "transferencia.csv", delimiter=",", skiprows=1)
copia = nuevo.copy()
salida = normalizar(nuevo)
error_transferencia = float(np.max(np.abs(salida - normalizar_lento(nuevo))))
assert salida.shape == nuevo.shape and np.isfinite(salida).all()
assert np.array_equal(nuevo, copia), "No debes modificar la entrada."
assert np.allclose(normalizar(np.array([[3., 7.]])), 0)
resultado["transferencia"] = error_transferencia
resultado["desviacion_constante"] = float(salida[:, 1].std())


## Mirar la idea
Compara el dibujo con lo que esperabas antes de ejecutar.


In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
for j in range(X.shape[1]):
    ejes[0].plot(X[:, j], marker="o", label=f"columna {j}")
    ejes[1].plot(obtenido[:, j], marker="o", label=f"columna {j}")
ejes[0].set(title="Antes: unidades y escalas distintas", xlabel="fila", ylabel="valor")
ejes[1].set(title="Después: media 0, desviación 1 si varía", xlabel="fila", ylabel="valor estandarizado")
for eje in ejes:
    eje.legend()
    eje.grid(alpha=0.15)
fig.tight_layout()
plt.show()


## Para seguir explorando
¿Qué información conserva normalizar una columna y cuál cambia? Prueba una columna constante, una sola fila o unidades cien veces mayores. Predice el resultado antes de ejecutarlo. Compara los tiempos por curiosidad; el factor depende del equipo.


## Comprobaciones del ejemplo
Estas aserciones detectan errores técnicos en el cuaderno y en su solución de referencia. No son un examen ni una escala de capacidad.


In [ ]:
assert isinstance(resultado, dict)


## Resultado reproducible
Esta celda guarda automáticamente las medidas para comprobar el material. Puedes conservar una copia de tu notebook y tus propias notas; no hay un formulario que rellenar.


In [ ]:
resultado.update({"laboratorio": '00_numpy', "version": 'inicio para completar',
                  "datos": "sintéticos CC0-1.0",
                  "metrica": 'error absoluto máximo frente al baseline correcto (menor es mejor)', "split": 'sin entrenamiento; desarrollo y transferencia son matrices separadas'})
Path("resultado.json").write_text(json.dumps(resultado, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")
print(json.dumps(resultado, ensure_ascii=False, indent=2))
